# Qwen3-TTS в Colab: один запуск — проверить всё и замерить RTF

Run all. Один прогон (~6–10 мин) делает сразу всё:

1. проверяет окружение (Python/glibc/GPU/ffmpeg) и останавливается, если GPU нет;
2. ставит пины `mlx[cuda]==0.32.3` + `mlx-audio==0.5.8` (linux-колёса, без сборки из исходников);
3. проверяет, что MLX реально считает на GPU (fp16 matmul + assert);
4. клонирует `davnozdu/qwen-epub-audiobook`, применяет к нему правки под Linux (не macOS) и **проверяет
   их результатом**: не осталось хардкодов Homebrew, requirements переписаны, семь скриптов компилируются,
   ни один этап кроме LLM-разметки не спрашивает stdin;
5. берёт образец голоса — ваш `narrator.wav` из диалога загрузки, а если диалог пропустить, то демо-образец
   из репозитория Qwen3-TTS (тогда RTF верный, а качество русского — нет);
6. синтезирует одну фразу, затем четыре, считает **RTF** (секунды синтеза на секунду звука);
7. печатает вердикт и сохраняет `metrics.json` рядом с тестовыми WAV + даёт скачать результат.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.

Единственное, что в этом прогоне остаётся неизвестным до его конца, — скорость (RTF). Всё остальное
(колёса, пути, установка весов, ключи) проверяется тут же, на месте.


## Шаг 1. Окружение

In [ ]:
import os

# Кэш свёрток MLX/CUDA: по умолчанию 128, и генерация падает на середине с
# «Cache thrashing is happening, please set the environment variable
#  MLX_CUDA_CONV_CACHE_SIZE to a larger value than 128».
# Это потолок, а не резервация: память берётся по мере надобности, поэтому на T4
# с её ~15 ГБ ставим 8192 МБ. Переменную читает первый импорт mlx — ставим до него.
#  Кэши MLX/CUDA считаются в записях, а не в мегабайтах: по умолчанию свёртки 128, граф 400,
#  внимание 256/64, FFT 128. Когда записей не хватает, MLX печатает «Cache thrashing is happening,
#  please set the environment variable <имя> to a larger value than <N>» и синтез падает на середине
#  книги. Ставим потолки с запасом (это лимит на число записей, а не резервация памяти).
CACHE_ENTRIES = {"MLX_CUDA_CONV_CACHE_SIZE": 12288, "MLX_CUDA_GRAPH_CACHE_SIZE": 8192,
                 "MLX_CUDA_SDPA_CACHE_SIZE": 4096, "MLX_CUDA_SDPA_BACKWARD_CACHE_SIZE": 1024,
                 "MLX_CUDA_FFT_CACHE_SIZE": 1024}
for _name, _value in CACHE_ENTRIES.items():
    os.environ.setdefault(_name, str(_value))

import platform, shutil, subprocess, sys

print("Python:", sys.version.split()[0])
print("glibc :", ".".join(platform.libc_ver()[1].split(".")[:2]) or "?")
print("ffmpeg:", shutil.which("ffmpeg") or "НЕ НАЙДЕН")

gpu, why = "", []
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
    gpu = out.stdout.strip()
    if not gpu:
        why.append((out.stderr or "nvidia-smi вернул пусто").strip()[:120])
except Exception as e:
    why.append(str(e))

if not gpu:                     # запасная проверка: CUDA через torch (в образе Colab он есть)
    try:
        import torch
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            gpu = f"{torch.cuda.get_device_name(0)} ({props.total_memory / 2 ** 30:.0f} ГБ, определён через torch)"
    except Exception as e:
        why.append(f"torch: {e}")

print("GPU   :", gpu or "НЕ НАЙДЕН")
if not gpu:
    print("  почему не нашли:", "; ".join(why) or "неизвестно")
    raise AssertionError(
        "GPU не найден. Runtime → Change runtime type → T4 GPU, затем Runtime → Restart session "
        "и Run all заново (смена рантайма перезапускает машину). Без GPU MLX-CUDA не поедет.")
print("  ориентир: T4 = sm_75, ~15 ГБ доступной памяти")
if "T4" not in gpu:
    print("  это не T4 — не страшно: тест продолжается, цифра будет относиться к вашей карте.")

## Шаг 2. Установка пинов

`mlx[cuda]` под Linux подставляет `mlx-cuda-12` (`manylinux_2_35_x86_64`) вместе с nvidia-колёсами
(cublas 12.9, cufft 11.4, nvrtc 12.9, cudnn 9, nccl, cusolver). Системный CUDA-тулкит не нужен.
`miniaudio` (нужен `mlx-audio` для чтения WAV) может собраться из исходников — это нормально, ~минута.


In [ ]:
%pip install -q "mlx[cuda]==0.32.3" "mlx-audio==0.5.8"
# если import mlx.core упадёт на libnvrtc.so.12 / libcublas.so.12:
# %pip install -q nvidia-cuda-nvrtc-cu12 nvidia-cublas-cu12

## Шаг 3. MLX видит GPU

In [ ]:
import importlib.metadata as md

import mlx.core as mx
import numpy as np

for pkg in ("mlx", "mlx-cuda-12", "mlx-audio"):
    try:
        print(f"{pkg:14s}", md.version(pkg))
    except Exception:
        print(f"{pkg:14s} НЕ УСТАНОВЛЕН" + ("   ← Metal-сборка, на Linux GPU не поедет"
                                            if pkg == "mlx-cuda-12" else ""))

device = mx.default_device()
on_gpu = "gpu" in str(device).lower()          # repr выглядит как «Device(gpu, 0)»
print("default device:", device, "| это GPU:", on_gpu)
try:
    print("device info   :", mx.device_info())
except Exception as e:
    print("device info   :", e)

a = mx.random.normal((1024, 1024), dtype=mx.float16)
b = mx.random.normal((1024, 1024), dtype=mx.float16)
s = float(mx.sum(a @ b))
print("fp16 matmul   :", f"{s:.1f}", "ok" if np.isfinite(s) else "NaN — плохой знак")
assert on_gpu, f"MLX выбрал CPU ({device}) — генерация будет бессмысленно медленной"
print("\n✓ GPU и MLX работают")

## Шаг 4. Репозиторий и правки под Linux + проверка правок

Клон публичный, токен не нужен. Дальше ровно те пять правок, которые и делают перенос возможным:
`mlx-metal` (существует только под macOS) → `mlx[cuda]`, и пять хардкодов `/opt/homebrew/bin/ffmpeg`
на системный. Проверка ниже — не «команда прошла», а результат: пустой список хардкодов, пины в
requirements, `py_compile` по семи скриптам и список интерактивных мест.


In [ ]:
%cd /content
!rm -rf qwen-epub-audiobook
!git clone -q https://github.com/davnozdu/qwen-epub-audiobook.git
%cd /content/qwen-epub-audiobook

# 1) macOS-only зависимость → CUDA-сборка MLX
!sed -i 's#mlx-metal==0.32.3#mlx[cuda]==0.32.3#' requirements.txt requirements-mlx.txt
# 2) пять хардкодов Homebrew ffmpeg/ffprobe → системный /usr/bin
!sed -i 's#/opt/homebrew/bin/#/usr/bin/#g' audiobook_epub.py normalize_book_audio.py mix_book_music.py synthesize_qwen_mlx.py

In [ ]:
import pathlib, py_compile

ROOT = pathlib.Path("/content/qwen-epub-audiobook")

# 1) хардкоды путей; setup_audiobook.py — маковый установщик, его мы не запускаем
IGNORED = {"setup_audiobook.py"}
left = []
for p in sorted(ROOT.glob("*.py")):
    if p.name in IGNORED:
        continue
    for i, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
        if "/opt/homebrew/bin" in line:
            left.append(f"{p.name}:{i}")
ignored = [f"{p.name}" for p in sorted(ROOT.glob("*.py"))
           if p.name in IGNORED and "/opt/homebrew/bin" in p.read_text(encoding="utf-8")]
print("хардкоды Homebrew:", left or "нет")
print("не запускаем (маковый установщик):", ignored or "нет")
assert not left, "остались маковыe пути"

# 2) пины
req = (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines()
mlx_lines = [l for l in req if l.strip().startswith("mlx")]
print("mlx в requirements.txt:", mlx_lines)
assert not any(v.startswith("mlx-metal") for v in mlx_lines), "mlx-metal на Linux не ставится"
assert any("mlx[cuda]" in v for v in mlx_lines), "нет extra cuda"

# 3) компилируемость после правок
scripts = ["audiobook_epub.py", "normalize_book_audio.py", "mix_book_music.py",
           "synthesize_qwen_mlx.py", "prepare_qwen.py", "prepare_silero.py", "prepare_speech.py"]
for name in scripts:
    py_compile.compile(str(ROOT / name), doraise=True)
print(f"py_compile: {len(scripts)} скриптов ок")

# 4) где спрашивают stdin/ключ (в Colab это EOFError)
for p in sorted(ROOT.glob("*.py")):
    hits = [f"{p.name}:{i}" for i, l in enumerate(p.read_text(encoding="utf-8").splitlines(), 1)
            if ("getpass" in l or l.strip().startswith("input(") or "= input(" in l)]
    if hits:
        print("интерактив →", ", ".join(hits))
print("\n✓ правки применены и проверены результатом")

## Шаг 5. Образец голоса

Всплывёт диалог — выберите `narrator.wav` из `~/Downloads/colab-ref/` (9,4 с, PCM float32, 24 кГц, моно)
вместе с текстом из `ref_text.txt`. Если диалог закрыть/отменить, тест пойдёт на демо-образце из
репозитория Qwen3-TTS: цифра RTF будет верной, а качество русского — нет.


In [ ]:
import os, urllib.request
from pathlib import Path

from google.colab import files

REF_AUDIO = ""
try:
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        # upload() пишет в текущий каталог (мы в папке репозитория), ищем и там, и в /content
        found = [p for p in (Path(name), Path.cwd() / name, Path("/content") / name) if p.exists()]
        REF_AUDIO = str(found[0].resolve()) if found else ""
except Exception as e:                       # диалог отменён / нет ответа
    print("загрузка пропущена:", e)

if not REF_AUDIO:
    for candidate in ("/content/narrator.wav", "/content/drive/MyDrive/narrator.wav"):
        if os.path.exists(candidate):
            REF_AUDIO = candidate
            break

FALLBACK = False
if not REF_AUDIO:
    REF_AUDIO = "/content/clone.wav"
    if not os.path.exists(REF_AUDIO):
        urllib.request.urlretrieve(
            "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav", REF_AUDIO)
    FALLBACK = True
    print("!! Демо-образец (английский): RTF померим, качество русского на нём плохое")

print("образец:", REF_AUDIO, os.path.getsize(REF_AUDIO), "байт")

In [ ]:
from mlx_audio.utils import load_audio

wave = load_audio(REF_AUDIO, sample_rate=24000)         # тот же загрузчик, что и в синтезе
sec = wave.shape[0] / 24000
print(f"{sec:.1f} с, {int(wave.shape[0])} сэмплов, пик {float(mx.max(mx.abs(wave))):.3f}")
assert 3 <= sec <= 25, "образец должен быть 5–15 с (допустимо 3–25)"

## Шаг 6. Текст: расшифровка образца и тестовые фразы

`REF_TEXT` — слово в слово то, что в образце. Тестовый текст — с ударениями в формате «+ вплотную
перед гласной» (скрипт переводит это в U+0301 после гласной; над «ё» ударение снимается).
Разметка проверяется `assert`-ом тут же, в ячейке.


In [ ]:
NARRATOR_REF_TEXT = 'Усмехнулся тоже и черномазый. Белокурый несколько удивился, что ему удалось сказать довольно, впрочем, плохой каламбур.'
DEMO_REF_TEXT = ("Okay. Yeah. I resent you. I love you. I respect you. "
                 "But you know what? You blew it! And thanks to you.")
REF_TEXT = DEMO_REF_TEXT if FALLBACK else NARRATOR_REF_TEXT
print("REF_TEXT:", REF_TEXT[:90], "…")

TEST_ONE = 'З+амок сто+ит на холм+е, а м+астер почин+ил зам+ок в двер+и. Князь Лев Никол+аевич М+ышкин верн+улся в Петерб+ург р+анним +утром и ср+азу прошёл в сво+ю кварт+иру на Лит+ейном просп+екте.'
TEST_FOUR = "\n".join([
    'З+амок сто+ит на холм+е, а м+астер почин+ил зам+ок в двер+и. Князь Лев Никол+аевич М+ышкин верн+улся в Петерб+ург р+анним +утром и ср+азу прошёл в сво+ю кварт+иру на Лит+ейном просп+екте.',
    'В нач+але апр+еля сто+яли +ясные мор+озные дни, и Н+евский просп+ект был ещё покр+ыт лёгким сух+им снег+ом. Прох+ожие тороп+ились, подним+ая вор+отники, а изв+озчики бран+ились на х+олод и гололёд.',
    'Он вошёл в небольшу+ю к+омнату с выс+окими +окнами и ув+идел на стол+е развёрнутый журн+ал, очк+и и ост+ывший чай. За +окном крич+али вор+обьи, и +этот прост+ой звук неожид+анно успок+оил ег+о.',
    'Д+ело б+ыло решен+о, и остав+алось т+олько подпис+ать бум+аги. Он д+олго смотр+ел на стр+оки, пот+ом аккур+атно слож+ил лист вч+етверо и убр+ал ег+о во вн+утренний карм+ан сюртук+а.',
])
print(f"\nодна фраза: {len(TEST_ONE)} симв.; четыре абзаца: {len(TEST_FOUR)} симв.")

import re as _re
_VOW = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"
for _name, _t in (("TEST_ONE", TEST_ONE), ("TEST_FOUR", TEST_FOUR)):
    _bad = _re.findall(r"\+(?![" + _VOW + "])", _t)
    assert not _bad, f"{_name}: «+» не перед гласной: {_bad}"
print("разметка ударений корректна")

## Шаг 7. Хелперы из `rab/qwen_tts_stress.py` (скопированы без изменений)

In [ ]:
import re

ACUTE = "́"
VOWELS = "аеёиоуыэюяАЕЁИОУЫЭЮЯ"


def normalize_stress(text: str) -> str:
    """«+» перед гласной → U+0301 после неё."""
    text = re.sub(rf"\+([{VOWELS}])", rf"\1{ACUTE}", text)
    return re.sub(rf"([ёЁ]){ACUTE}", r"\1", text)


def split_text(text: str, max_chars: int) -> list:
    """Режет текст на куски по абзацам и предложениям, не длиннее max_chars."""
    chunks = []
    for para in text.splitlines():
        para = " ".join(para.split())
        if not para:
            continue
        cur = ""
        for sent in re.split(r"(?<=[.!?…»])\s+", para):
            while len(sent) > max_chars:
                cut = max(sent.rfind(", ", 0, max_chars), sent.rfind(" ", 0, max_chars))
                cut = cut + 1 if cut > 0 else max_chars
                if cur:
                    chunks.append(cur)
                    cur = ""
                chunks.append(sent[:cut].strip())
                sent = sent[cut:].strip()
            if cur and len(cur) + 1 + len(sent) > max_chars:
                chunks.append(cur)
                cur = sent
            else:
                cur = f"{cur} {sent}".strip()
        if cur:
            chunks.append(cur)
    return chunks


print("кусков в тесте:", len(split_text(normalize_stress(TEST_FOUR), 250)))

## Шаг 8. Модель (≈4,6 ГБ, первый раз 2–4 мин)

In [ ]:
import time

from mlx_audio.tts.utils import load_model

MODEL = "siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf"
MODEL_FILES = ["config.json", "generation_config.json", "preprocessor_config.json",
               "tokenizer_config.json", "vocab.json", "merges.txt",
               "model.safetensors", "speech_tokenizer/*"]

t0 = time.time()
model = load_model(MODEL, allow_patterns=MODEL_FILES)
print(f"модель загружена за {time.time() - t0:.0f} с, sample_rate={model.sample_rate}")

t0 = time.time()
ref_wave = load_audio(REF_AUDIO, sample_rate=model.sample_rate)
print(f"образец закодирован за {time.time() - t0:.1f} с → {tuple(ref_wave.shape)}")

## Шаг 9. Одна фраза → RTF

In [ ]:
import json
from pathlib import Path

from IPython.display import Audio, display
from scipy.io import wavfile

OUT = Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
sr = model.sample_rate


def synth(chunk, seed=42):
    mx.random.seed(seed)
    t = time.time()
    parts = [np.asarray(r.audio, dtype=np.float32)
             for r in model.generate(text=chunk, lang_code="russian",
                                     ref_audio=ref_wave, ref_text=REF_TEXT, temperature=0.9)]
    wav = np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)
    return wav, time.time() - t


def save(path, wav):
    wavfile.write(str(path), sr, (np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16))


phrase = normalize_stress(TEST_ONE)
wav, dt = synth(phrase, seed=43)
sec = len(wav) / sr
print("фраза:", phrase[:70], "…")
print(f"аудио {sec:.1f} с за {dt:.1f} с → RTF {dt / sec:.2f} ({sec / dt:.1f}× быстрее реального времени)")
save(OUT / "one.wav", wav)
display(Audio(str(OUT / "one.wav")))

## Шаг 10. Четыре фразы → устойчивый RTF и проекция

In [ ]:
chunks = split_text(normalize_stress(TEST_FOUR), 250)
print(f"кусков: {len(chunks)}\n")

rows, pieces = [], []
for i, chunk in enumerate(chunks, 1):
    wav, dt = synth(chunk, seed=42 + i)
    sec = len(wav) / sr
    rows.append({"i": i, "chars": len(chunk), "audio_s": round(sec, 2),
                 "synth_s": round(dt, 2), "rtf": round(dt / sec, 3)})
    print(f"[{i}/{len(chunks)}] {len(chunk):3d} симв. → {sec:5.1f} с звука за {dt:5.1f} с: RTF {dt / sec:.2f}")
    pieces.append(wav)
    pieces.append(np.zeros(int(sr * 0.35), dtype=np.float32))

total_audio = sum(r["audio_s"] for r in rows)
total_synth = sum(r["synth_s"] for r in rows)
rtf_mean = total_synth / total_audio
print(f"\nитого: {total_audio:.1f} с звука за {total_synth:.1f} с → RTF {rtf_mean:.2f}")
print(f"книга 10 ч звука: ≈ {10 * rtf_mean:.1f} ч синтеза; в сессию 12 ч влезает ≈ {11 / rtf_mean:.1f} ч звука")

save(OUT / "four.wav", np.concatenate(pieces))
metrics = {"model": MODEL, "device": str(mx.default_device()), "ref_audio": REF_AUDIO,
           "demo_reference": FALLBACK, "chunks": rows, "rtf_weighted": round(rtf_mean, 3),
           "book_10h_synth_hours": round(10 * rtf_mean, 1),
           "max_audio_hours_per_session": round(11 / rtf_mean, 1)}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
display(Audio(str(OUT / "four.wav")))

## Шаг 11. Вердикт

In [ ]:
rtf = metrics["rtf_weighted"]
if rtf < 0.3:
    verdict = f"отлично: книга 10 ч ≈ {10 * rtf:.1f} ч синтеза — полный пайплайн едет в Colab"
elif rtf < 0.7:
    verdict = f"рабочо: книга 10 ч ≈ {10 * rtf:.1f} ч; за сессию влезает ≈ {11 / rtf:.1f} ч звука"
elif rtf < 1.0:
    verdict = (f"на грани: книга 10 ч ≈ {10 * rtf:.1f} ч, за сессию только ≈ {11 / rtf:.1f} ч звука — "
               "делить книгу по progress.json")
else:
    verdict = (f"плохо: {rtf:.2f}× медленнее реального времени, за 12 ч сессии только ≈ {11 / rtf:.1f} ч звука — "
               "нужен A100/L4 или план Б (torch)")
print(f"RTF = {rtf:.2f}\nВЕРДИКТ: {verdict}")

if metrics["demo_reference"]:
    print("\nнапоминание: образец был демо (английский) — качество русского не проверено, только скорость")

## Шаг 12. Забрать результат

Обе дорожки (MLX-тест выше и torch-вариант из соседнего ноутбука) считают один и тот же RTF на одной
карте — сравнение даёт ответ, какой бэкенд брать в продакшн.


In [ ]:
from google.colab import files

files.download(str(OUT / "four.wav"))
files.download(str(OUT / "metrics.json"))

## Что дальше — полный пайплайн (когда RTF устроил)

```bash
# в /content/qwen-epub-audiobook (уже склонирован и пропатчен в шаге 4)
python -c "import mlx.core as mx; print(mx.default_device())"        # ждём gpu
python prepare_silero.py --manifest <manifest.json>                    # ударения: словарь + Silero
python prepare_qwen.py run <book.epub> -o <out-dir>                     # роли; ТУТ нужен ключ Ollama
python synthesize_qwen_mlx.py <manifest.json>                           # синтез, resume по progress.json
python normalize_book_audio.py / mix_book_music.py                      # сведение
```

Ключ нужен ровно на одной стадии — LLM-разметке (`prepare_qwen.py run`, `prepare_speech.py run`).
Синтез ключа не требует вообще; `prepare_silero.py` по умолчанию (`--llm-disputes off`) не спрашивает,
а при `ask` ловит `EOFError` и идёт без LLM; `prepare_speech.py offline` — тоже без LLM.
Пройти без ключа можно командами `advance` (пишут `request.json`, запрос выполняешь где угодно) и `offline`.
Ключ в Colab кладётся в окружение: `os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")`.

Не запускать: `setup_audiobook.py` (маковый установщик), `build_audiobook.py` (интерактивный: три
`input()` + ключ), `*.command` (зовут `/opt/homebrew/opt/python@3.11`).

Выход держать на Google Drive: 12-часовой сессии на книгу не хватит, а `progress.json` и sha256
сегментов позволяют продолжить синтез в следующем запуске.
